In [1]:
import collections as col
import pathlib as pl
import pickle as pck
import re

import pandas as pd

cfg_nb = pl.Path("../../load-config.ipynb").resolve(strict=True)
%run $cfg_nb

_NB_SESSION = __session__
NB_NAME = pl.Path(_NB_SESSION).name
NB_PATH = pl.Path(_NB_SESSION).parent
NB_REL_PATH = NB_PATH.relative_to(CONFIG["project_repo"]).joinpath(NB_NAME)


glob_pattern = "**/*refOriented*fasta.gz"

search_path = CONFIG["local_hilbert_prefix"].joinpath(CONFIG["hilbert_verkko_assembly_folder"])

sample_annotation = CONFIG["project_repo"].joinpath(
    "annotation", "norm", "samples_1kg_pedigree.ext.tsv"
).resolve(strict=True)

haplogroup_annotation = CONFIG["project_repo"].joinpath(
    "annotation", "raw", "hallast_2023_nature.supp-table-1.sample-desc.tsv"
).resolve(strict=True)

haplogroups = []
hgrouped_samples = set()
with open(haplogroup_annotation, "r") as listing:
    for line in listing:
        columns = line.strip().split()
        sample = columns[1]
        haplogroup = columns[2]
        if sample == "T2T-Y":
            sample = "NA24385"
            haplogroup = columns[3]
        hg_root = haplogroup[:2]
        haplogroups.append((sample, haplogroup.strip(), hg_root))
        hgrouped_samples.add(sample)

# 2025-01-31
# add new haplogroup information from Pille's annotation
haplogroup_annotation = CONFIG["project_repo"].joinpath(
    "annotation", "raw", "20250130_verkko_assemblies.no-fp.PH.tsv"
).resolve(strict=True)
with open(haplogroup_annotation, "r") as listing:
    # skip header
    _ = listing.readline()
    for line in listing:
        columns = line.strip().split()
        if columns[1] == "female":
            continue
        if columns[0] in hgrouped_samples:
            continue
        sample, _, _, hg_short, haplogroup, _, _, _ = line.strip().split()
        haplogroups.append((sample, haplogroup.strip(), hg_short.strip()))

# 2025-03-14
# manual add haplogroups for new samples ASK/father
haplogroups.append(
    ("NA24149", "J1a2a1a2c1a1", "J1")
)
haplogroups.append(
    ("NA24631", "D1a1a1a", "D1")
)

haplogroups = pd.DataFrame.from_records(haplogroups, columns=["sample", "haplogroup", "hg_short"])
haplogroups.set_index("sample", inplace=True)

samples = pd.read_csv(sample_annotation, sep="\t", header=0, comment="#")
samples.set_index("individual_id", inplace=True)

asm_norm = {
    "verkko-hi-c": "vrk-hic",
    "verkko-thic": "vrk-thc"
}

au_norm = {
    "haplotype1": "hap1",
    "haplotype2": "hap2",
    "": "wg"
}

norm_sample = {
    "HG002": "NA24385",
    "HG005": "NA24631",
    "NA24631": "HG005",
    "NA24385": "HG002",
    "HG003": "NA24149",
    "NA24149": "HG003",
    "NA24143": "HG004",
    "HG004": "NA24143"
}

fasta_file_paths = None
t2t_contigs = None
t2t_scaffolds = None

cache_file_path = prep_cache_file(NB_REL_PATH, "verkko_assemblies_hilbert.pkl")
if not cache_file_path.is_file():
    assert search_path.is_dir(), f"Search path unavailable: {search_path}"
    
    fasta_file_paths = []
    t2t_contigs = dict()
    t2t_scaffolds = dict()
    for fasta_file in search_path.glob(glob_pattern):
        sample = fasta_file.name.split(".")[0]
        t2t_contig_file = fasta_file.parent.joinpath(sample + ".t2t_ctgs").resolve(strict=True)
        t2t_scaffold_file = fasta_file.parent.joinpath(sample + ".t2t_scfs").resolve(strict=True)
        fasta_file_paths.append(fasta_file)
        t2t_contigs[sample] = t2t_contig_file
        t2t_scaffolds[sample] = t2t_scaffold_file
    cache_dump = {
        "fasta_files": fasta_file_paths,
        "t2t_contig_files": t2t_contigs,
        "t2t_scaffold_files": t2t_scaffolds
    }
    with open(cache_file_path, "wb") as cache:
        _ = pck.dump(cache_dump, cache)
else:
    with open(cache_file_path, "rb") as cache:
        cache_dump = pck.load(cache)
        fasta_file_paths = cache_dump["fasta_files"]
        t2t_contigs = cache_dump["t2t_contig_files"]
        t2t_scaffolds = cache_dump["t2t_scaffold_files"]

# data cached or cached data loaded

match_assembly_type = re.compile(
    "(refOriented.fasta|refOriented.haplotype1.fasta|refOriented.haplotype2.fasta)"
)

rows = col.defaultdict(dict)
for fasta_file in fasta_file_paths:
    sample = fasta_file.name.split(".")[0]
    sample = norm_sample.get(sample, sample)
    assembly_type_key = match_assembly_type.search(fasta_file.name)
    assert assembly_type_key is not None
    astk = assembly_type_key.group(0)
    astk = astk.strip(".fasta").strip("refOriented.")
    assembly_type = asm_norm[fasta_file.parent.parent.name]
    try:
        asm_unit = au_norm[astk]
    except KeyError:
        print(fasta_file.name.split("."))
        raise
    try:
        sample_sex = samples.at[sample, "karyotype"]
    except KeyError:
        print("Missing: ", sample)
        continue
    rows[sample]["sample_sex"] = sample_sex
    rows[sample][f"asm_{asm_unit}"] = replace_path_prefix(fasta_file)
    rows[sample]["assembly_type"] = assembly_type

    # auxiliary files
    try:
        t2t_contig_file = t2t_contigs[sample]
        t2t_scaffold_file = t2t_scaffolds[sample]
    except KeyError:
        original_sample = norm_sample[sample]
        t2t_contig_file = t2t_contigs[original_sample]
        t2t_scaffold_file = t2t_scaffolds[original_sample]
    rows[sample]["t2t_ctg"] = replace_path_prefix(t2t_contig_file)
    rows[sample]["t2t_scf"] = replace_path_prefix(t2t_scaffold_file)

assemblies = pd.DataFrame.from_dict(rows, orient="index")
assemblies.index.name = "sample"
assemblies.sort_index(inplace=True)
assemblies = assemblies[["sample_sex", "assembly_type", "asm_hap1", "asm_hap2", "asm_wg", "t2t_ctg", "t2t_scf"]]

assemblies = assemblies.join(haplogroups)

select_females = assemblies["sample_sex"] == "female"
select_males = assemblies["sample_sex"] == "male"
assemblies.loc[select_females, "haplogroup"] = "XX"
assemblies.loc[select_females, "hg_short"] = "XX"

assemblies["study_generation"] = 2025

# known from Hallast et al. 2023
select_known_males = assemblies.index.isin(hgrouped_samples)

assemblies.loc[select_known_males & select_males, "study_generation"] = 2023
assemblies["haplogroup"] = assemblies["haplogroup"].fillna("unknown", inplace=False)
assemblies["hg_short"] = assemblies["hg_short"].fillna("UN", inplace=False)

assemblies = assemblies[
    [
        "sample_sex", "assembly_type", "hg_short", "haplogroup",
        "study_generation", "asm_hap1", "asm_hap2", "asm_wg", "t2t_ctg", "t2t_scf"
    ]
]

sample_sheet_file = CONFIG["project_repo"].joinpath("samples", "verkko_assemblies.tsv")
sample_sheet_file.parent.mkdir(exist_ok=True, parents=True)

with open(sample_sheet_file, "w") as table:
    _ = table.write(f"# {TIMESTAMP}\n")
    _ = table.write(f"# N={assemblies.shape[0]}\n")
    assemblies.to_csv(table, sep="\t", header=True, index=True)
    